# 02 · Build a Server

**Module 11 · MCP Tool Servers**

We turn the shop's tools into an MCP server. This is the part you would build at work: the **server side**. It is much shorter than you expect.

> This notebook makes no OpenAI calls. A server never talks to a model.

In [1]:
import json

from mcp import Client

import shop_server
from toolbox import Toolbox, run_sync

print("Ready")

Ready


## 1. A tool is a Python function
Three things in a plain function become the three parts of a tool:

| In the function | Becomes |
|---|---|
| its **name** | the tool's name |
| its **docstring** | the tool's **description**, which is what the model reads. Write it for the model |
| its **type hints** (`order_id: str`) | the **input schema**. You write no JSON by hand |

The decorator `@mcp.tool()` registers the function. Here are the shop's tools:

*From `shop_server.py`:*

```python
@mcp.tool(annotations=ToolAnnotations(read_only_hint=True))
def lookup_order(order_id: str) -> str:
    """Get the details of one order: item, price, status and delivery date. The order id looks like A1001."""
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return "No order found with id " + order_id + "."

    details = dict(order)
    if order["status"] == "delivered":
        details["days_since_delivery"] = (date.today() - date.fromisoformat(order["delivered_on"])).days
    return json.dumps(details)


@mcp.tool(annotations=ToolAnnotations(read_only_hint=True))
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order can be refunded under the shop's policy (30 days after delivery)."""
    allowed, explanation = refund_rules(order_id)
    return explanation


@mcp.tool(annotations=ToolAnnotations(read_only_hint=False, destructive_hint=True))
def issue_refund(order_id: str) -> str:
    """Issue a refund for an order. Only use this after the order is confirmed eligible.
    Refunds above $100 are not issued: they wait for a person to approve them."""
    order_id = order_id.strip().upper()
    allowed, explanation = refund_rules(order_id)
    if not allowed:
        return "Refund refused. " + explanation

    for refund in REFUNDS:
        if refund["order_id"] == order_id:
            return "A refund for order " + order_id + " was already issued."

    amount = ORDERS[order_id]["price"]
    if amount > AUTO_REFUND_LIMIT:          # the rule is in code, so no prompt can talk the server out of it
        if order_id not in PENDING:
            PENDING.append(order_id)
        return "The refund of $" + format(amount, ".2f") + " is above the automatic limit and now waits for a person. It has NOT been issued."

    REFUNDS.append({"order_id": order_id, "amount": amount})
    return "Refund of $" + format(amount, ".2f") + " issued for order " + order_id + "."


@mcp.tool(annotations=ToolAnnotations(read_only_hint=True))
def search_policies(question: str) -> str:
    """Search the shop's policies (shipping, returns, warranty, payments). Ask about one topic at a time."""
    words = re.findall(r"[a-z0-9$]+", question.lower())
    scored = []
    for file in sorted(KNOWLEDGE_DIR.glob("*.txt")):
        for paragraph in re.split(r"\n\s*\n", file.read_text(encoding="utf-8")):
            text = paragraph.strip().lower()
            score = 0
            for word in words:
                if len(word) > 3 and word in text:
                    score = score + 1
            if score > 0:
                scored.append((score, paragraph.strip()))
    scored.sort(reverse=True)

    lines = []
    for score, paragraph in scored[:3]:
        lines.append(paragraph)
    if len(lines) == 0:
        return "No policy found for that question."
    return "\n".join(lines)


@mcp.tool(annotations=ToolAnnotations(read_only_hint=True))
def list_refunds() -> str:
    """For staff and tests: the refunds that were issued, and the ones waiting for a person."""
    return json.dumps({"issued": REFUNDS, "waiting_for_a_person": PENDING})
```

Four design choices in that code are worth copying:

1. **The $100 limit is in the server, in code.** No prompt, in any app, can talk the server out of it.
2. **Errors are returned as text the model can use** ("Refund refused. Not eligible: ..."), not as crashes.
3. **Few tools, each with one clear job**, and descriptions that say *when* to use them ("Only use this after the order is confirmed eligible").
4. **Annotations** (`read_only_hint`, `destructive_hint`) tell a client what kind of tool this is. They are **hints**: a client must never trust them from a server it does not control (notebook 4).

## 2. Test the server without a model
A big advantage of MCP: the server can be tested **on its own**, with no agent and no language model. A test is just a client. Here we connect to the server **in the same process** (no child process needed) and call the tools:

In [2]:
async def try_the_tools():
    async with Client(shop_server.mcp) as client:
        listed = await client.list_tools()
        names = []
        for tool in listed.tools:
            names.append(tool.name)
        print("Tools:", names)

        print()
        print("What the model will read about issue_refund:")
        for tool in listed.tools:
            if tool.name == "issue_refund":
                print("  description:", tool.description.replace("\n", " "))
                print("  input schema:", json.dumps(tool.input_schema["properties"]))

        print()
        for order_id in ["A1001", "A1003", "A1004"]:
            result = await client.call_tool("issue_refund", {"order_id": order_id})
            print(order_id, "->", result.content[0].text)

        result = await client.call_tool("list_refunds", {})
        print()
        print("State of the shop:", result.content[0].text)


shop_server.REFUNDS.clear()
shop_server.PENDING.clear()
run_sync(try_the_tools)

Tools: ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'search_policies', 'list_refunds']

What the model will read about issue_refund:
  description: Issue a refund for an order. Only use this after the order is confirmed eligible. Refunds above $100 are not issued: they wait for a person to approve them.
  input schema: {"order_id": {"title": "Order Id", "type": "string"}}



A1001 -> Refund of $79.00 issued for order A1001.
A1003 -> Refund refused. Not eligible: delivered 80 days ago, the refund window is 30 days.
A1004 -> The refund of $250.00 is above the automatic limit and now waits for a person. It has NOT been issued.

State of the shop: {"issued": [{"order_id": "A1001", "amount": 79.0}], "waiting_for_a_person": ["A1004"]}


The refund for A1001 ($79) was issued, A1003 was refused (80 days old), and A1004 ($250) is waiting for a person. That is a unit test for the whole tool layer, and it ran in a fraction of a second at zero cost. **Write tests like this for every server you build.**

## 3. Resources and prompts
Tools are for the **model** to call. Two other things a server can offer are meant for the **app**:

*From `shop_server.py`:*

```python
@mcp.resource("shop://policies")
def all_policies() -> str:
    """The full text of the shop's policies."""
    text = ""
    for file in sorted(KNOWLEDGE_DIR.glob("*.txt")):
        text = text + file.read_text(encoding="utf-8").strip() + "\n\n"
    return text.strip()


@mcp.resource("order://{order_id}")
def order_details(order_id: str) -> str:
    """One order, as data. The {order_id} in the address makes this a resource TEMPLATE."""
    return lookup_order(order_id)


@mcp.prompt()
def refund_reply(order_id: str, customer_name: str) -> str:
    """A ready-made prompt that asks for a short, kind reply about a refund."""
    return ("Write a reply of at most 3 short sentences to " + customer_name + " about the refund request for order "
            + order_id + ". Use the tools to check the order and the refund rules first.")
```

In [3]:
async def look_at_resources():
    async with Client(shop_server.mcp) as client:
        resources = await client.list_resources()
        for resource in resources.resources:
            print("Resource         :", resource.uri)

        templates = await client.list_resource_templates()
        for template in templates.resource_templates:
            print("Resource template:", template.uri_template)

        policies = await client.read_resource("shop://policies")
        print()
        print("shop://policies (the first 200 characters):")
        print(policies.contents[0].text[:200])

        order = await client.read_resource("order://A1002")
        print()
        print("order://A1002 ->", order.contents[0].text)

        prompt = await client.get_prompt("refund_reply", {"order_id": "A1001", "customer_name": "Maya"})
        print()
        print("The prompt template 'refund_reply' gives the app this message:")
        print(prompt.messages[0].content.text)


run_sync(look_at_resources)

Resource         : shop://policies
Resource template: order://{order_id}

shop://policies (the first 200 characters):
Payment methods: We accept credit cards, debit cards and UPI.

Cancelling an order: You can cancel an order for free until it ships. After it ships, you need to return it instead.

Changing the delive

order://A1002 -> {"item": "Running shoes", "price": 120.0, "status": "in_transit", "expected_delivery": "2026-11-26"}

The prompt template 'refund_reply' gives the app this message:
Write a reply of at most 3 short sentences to Maya about the refund request for order A1001. Use the tools to check the order and the refund rules first.


- A **resource** is data with an address. `order://{order_id}` is a **template**: the app fills in the id. The *app* decides when to read a resource (for example, to show the customer's order next to the chat). The model does not call it.
- A **prompt** is a reusable template, which a user can pick from a menu.

## 4. Over a real pipe
Tests in the same process are fast. But a real client starts the server as a **separate program**. Here is our own client library doing exactly that:

In [4]:
async def over_a_pipe():
    async with Toolbox() as box:
        await box.add_server("shop", "shop_server.py", trusted=True)
        print("Tools discovered over stdio:")
        for tool in box.tools:
            print("  ", tool.name)

        print()
        print(await box.call("lookup_order", {"order_id": "A1004"}))


run_sync(over_a_pipe)

Tools discovered over stdio:
   lookup_order
   check_refund_eligibility
   issue_refund
   search_policies
   list_refunds

{"item": "4K television", "price": 250.0, "status": "delivered", "delivered_on": "2026-11-17", "days_since_delivery": 5}


## 5. Rules for writing servers
- **Never `print()` in a stdio server.** Standard output **is** the protocol. A stray print corrupts it. Log to **stderr**.
- **Descriptions are prompts.** The model decides from them. Vague descriptions give wrong tool choices.
- **Keep secrets out of tool results.** Whatever a tool returns goes in front of the model.
- **Validate inside the tool**, because the model's arguments are untrusted input (Module 05).
- **Make writing tools safe to repeat** (idempotent, Module 08). Agents retry.

## What you learned
- A server is a handful of **decorated Python functions**: name, docstring and type hints become the tool definition.
- You can **test a server with no model**, using an MCP client in the same process.
- **Resources** are for the app, **prompts** are for the user, **tools** are for the model.
- The business rules belong **in the server**, in code.

**Next:** use this server from an agent, two different ways.